In [1]:
print("Downloading engine modules from public repository...")

BASE = "https://raw.githubusercontent.com/Denis2054/Context-Engineering-for-Multi-Agent-Systems/main/commons/engine"
modules = ["utils.py", "helpers.py", "agents.py", "registry.py", "engine.py"]

for m in modules:
    !curl -Lfs {BASE}/{m} --output {m}
    print(f"  ✅ {m}")

print("\n✅ All engine modules downloaded successfully.")


  ✅ utils.py
  ✅ helpers.py
  ✅ agents.py
  ✅ registry.py
  ✅ engine.py

✅ All engine modules downloaded successfully.


In [2]:
import utils

# Install all required packages (openai, pinecone, gradio, markdown, etc.)
utils.install_dependencies()

# Initialize OpenAI and Pinecone clients using API keys from Colab Secrets / env vars
client, pc = utils.initialize_clients()

# Install Gradio (not part of the standard engine deps)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
print("✅ Gradio installed.")

🚀 Installing required packages...
✅ All packages installed successfully.

🔑 Initializing API clients...
   - OpenAI client initialized.
   - Pinecone client initialized.
✅ Clients initialized successfully.
✅ Gradio installed.


In [3]:
import helpers        # LLM calls, embeddings, Pinecone search, moderation
import agents         # Librarian, Researcher, Writer agents
from registry import AGENT_TOOLKIT   # Agent capability registry
from engine import context_engine    # Main orchestrator

print("✅ All engine modules imported successfully.")


✅ All engine modules imported successfully.


In [4]:
import json
import html
import markdown as md_lib


def render_trace_dashboard(trace) -> str:
    """
    Converts an engine Trace object to a self-contained HTML string.
    Returns the HTML so Gradio can inject it into gr.HTML().
    """
    css = """

    """

    status_cls = "ok" if trace.status == "Success" else "fail"

    parts = [css, f"""



          Context Engine Trace
          "{html.escape(trace.goal)}"


          {trace.status}
          ⏱ {trace.duration:.2f}s


      Execution Workflow
    """]

    def _md(text):
        return md_lib.markdown(str(text)) if text else "No content recorded."

    for step in trace.steps:
        try:
            resolved_ctx = json.dumps(step.get("resolved_context", {}), indent=2)
        except Exception:
            resolved_ctx = str(step.get("resolved_context", "N/A"))

        output_raw = step.get("output", "N/A")
        if isinstance(output_raw, dict):
            rendered = ""
            for key in ["summary", "answer_with_sources", "answer", "output", "content"]:
                if key in output_raw and isinstance(output_raw[key], str):
                    rendered = f"{_md(output_raw[key])}"
                    break
            if not rendered:
                rendered = f"{html.escape(json.dumps(output_raw, indent=2))}"
        else:
            rendered = f"{_md(output_raw)}"

        t_in   = step.get("tokens_in",  "??")
        t_out  = step.get("tokens_out", "??")
        t_save = step.get("tokens_saved", 0)
        save_pill = f"📉 SAVED: {t_save}" if t_save else ""

        parts.append(f"""



              STEP {step['step']}
              {step['agent']}

                📥 IN: {t_in}
                📤 OUT: {t_out}
                {save_pill}


            OPEN LOGS



              Input Context (State)


                  ▶ View Resolved Source Data

                {html.escape(resolved_ctx)}


            Agent Output
            {rendered}


        """)

    if trace.final_output:
        fc = trace.final_output
        if isinstance(fc, dict):
            fc = fc.get("summary", fc.get("content", str(fc)))
        parts.append(f"""

          Final Orchestration Result
          {_md(fc)}

        """)

    parts.append("")
    return "".join(parts)

In [5]:
import engine
import json
import logging
import pprint
from helpers import call_llm_robust


# ── Robust Planner Patch ────────────────────────────────────────────────────

def planner_robust_patch(goal, capabilities, client, generation_model):
    """
    Enforces strict JSON schema in the planner prompt to prevent malformed plans
    on complex multi-step goals.
    """
    logging.info("[Planner-Patch] Generating strict-schema plan...")

    system_prompt = f"""
You are the strategic core of the Context Engine. Analyze the user's GOAL and create a step-by-step EXECUTION PLAN.

AVAILABLE CAPABILITIES
---
{capabilities}
---

INSTRUCTIONS:
1. Output MUST be a single JSON object with a "plan" key containing a list of step objects.
2. CRITICAL: Every step MUST strictly follow this schema:
   {{
      "step": ,
      "agent": "",
      "input": {{
          "": ""
      }}
   }}
   Agent arguments MUST be inside the "input" object, NOT at the root.
3. Use Context Chaining: write "
STEP_N_OUTPUT
" for values from previous steps.
"""
    try:
        raw = call_llm_robust(
            system_prompt, goal,
            client=client,
            generation_model=generation_model,
            json_mode=True
        )
        data = json.loads(raw)
        if "plan" not in data and isinstance(data, list):
            return data
        return data["plan"]
    except Exception as e:
        logging.error(f"Planner failed: {e}")
        raise


# Apply the patch
engine.planner = planner_robust_patch
print("✅ Engine patched with Robust Planner.")


# ── Main Execution Function ─────────────────────────────────────────────────

def execute_and_display(goal: str, config: dict, moderation_active: bool) -> str:
    """
    Runs the Context Engine and returns a rendered HTML trace string.
    Called by the Gradio button handler.
    """
    if not goal.strip():
        return "⚠️ Please enter a goal before running the engine."

    # Pre-flight moderation
    if moderation_active:
        report = helpers.helper_moderate_content(text_to_moderate=goal, client=client)
        if report.get("flagged"):
            return "🛑 Goal failed pre-flight moderation. Execution halted."

    try:
        result, trace = context_engine(goal, client=client, pc=pc, **config)
    except Exception as e:
        return f"🛑 Engine error: {html.escape(str(e))}"

    # Post-flight moderation
    if result and moderation_active:
        text_to_check = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
        report = helpers.helper_moderate_content(text_to_moderate=text_to_check, client=client)
        if report.get("flagged"):
            flagged_msg = "[Content flagged by moderation policy and redacted.]"
            if trace:
                trace.final_output = flagged_msg

    if trace:
        return render_trace_dashboard(trace)

    return "Engine failed to produce a trace."


print("✅ Engine Room ready.")

✅ Engine patched with Robust Planner.
✅ Engine Room ready.


In [6]:
import html  # needed by execute_and_display

# ── Control Deck Configuration ──────────────────────────────────────────────
# Modify these values to point to your own Pinecone index and preferred models.
config = {
    "index_name":          "genai-mas-mcp-ch3",
    "generation_model":    "gpt-5.1",
    "embedding_model":     "text-embedding-3-small",
    "namespace_context":   "ContextLibrary",
    "namespace_knowledge": "KnowledgeStore",
}

print("✅ Control Deck configuration loaded.")
print(f"   Index:            {config['index_name']}")
print(f"   Generation model: {config['generation_model']}")
print(f"   Embedding model:  {config['embedding_model']}")


✅ Control Deck configuration loaded.
   Index:            genai-mas-mcp-ch3
   Generation model: gpt-5.1
   Embedding model:  text-embedding-3-small


In [7]:
import gradio as gr

# ── Preset Goals ─────────────────────────────────────────────────────────────
PRESETS = {
    "── Select a preset goal ──": "",
    "[Marketing] 1. Quantum Drive Query": \
        "Summarize the key points of the QuantumDrive",
    "[Marketing] 2. ChronoTech Press Release Analysis (with citations)": \
        "Analyze the ChronoTech press release and summarize their core product "
        "messaging and value proposition. Please cite your sources.",
    "[Marketing] 3. Persuasive Brand Pitch": \
        "Write a persuasive pitch on our brand tone and voice guide",
    "[Legal] 4. NDA Retrieval & Summary": \
        "First, retrieve the content of the Non-Disclosure Agreement (NDA) from "
        "the knowledge base. Then, summarize its key points.",
    "[Legal] 5. Service Agreement — High-Fidelity RAG (with citations)": \
        "What are the key confidentiality obligations in the Service Agreement v1, "
        "and what is the termination notice period? Please cite your sources.",
    "[Legal] 6. Sanitization Limit Test": \
        "What did Mr. Smith advise his client regarding the assets?",
}


# ── Gradio Handler Functions ──────────────────────────────────────────────────

def load_preset(preset_name: str):
    """Fills the goal textbox when a preset is selected from the dropdown."""
    return PRESETS.get(preset_name, "")


def run_engine(goal: str, moderation: bool):
    """
    Gradio button handler. Yields intermediate status then the final HTML trace.
    Using yield turns this into a streaming generator so Gradio can show a
    'Running...' message before the (potentially slow) engine response arrives.
    """
    if not goal.strip():
        yield (
            "",
            ""
            "⚠️ Please enter a goal before running the engine."
        )
        return

    # Show a running indicator immediately
    yield (
        "⏳ **Context Engine running...** Multi-agent orchestration in progress.",
        ""
    )

    html_result = execute_and_display(goal, config, moderation)

    yield ("", html_result)   # clear status, show result


# ── Custom CSS ────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
#run-btn { background: #22543d !important; color: white !important;
           font-size: 1.1rem !important; font-weight: 900 !important;
           border-radius: 8px !important; }
#run-btn:hover { background: #276749 !important; }
#clear-btn { font-weight: 700 !important; border-radius: 8px !important; }
.gr-form { border-radius: 12px !important; }
.gradio-container { max-width: 1200px !important; margin: auto !important; }
"""


# ── Interface Definition ──────────────────────────────────────────────────────
with gr.Blocks(
    title="Universal Context Engine",
    theme=gr.themes.Default(primary_hue="green", neutral_hue="slate"),
    css=CUSTOM_CSS
) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.Markdown("""
    # 🧠 Universal Context Engine
    ### *Context Engineering for Multi-Agent Systems* — Standalone Web UI

    A **domain-agnostic Glass Box** multi-agent reasoning engine.
    Zero hard-coded business rules. Behavior is driven entirely by **context**.

    Select a preset goal or type your own below, then click **Run Engine**.
    """)

    gr.Markdown("---")

    # ── Input Zone ────────────────────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("### Control Deck")

            preset_dd = gr.Dropdown(
                label="Load Preset Goal",
                choices=list(PRESETS.keys()),
                value="── Select a preset goal ──",
                interactive=True,
                info="Select a preset to auto-fill the goal box, or type directly."
            )

            goal_box = gr.Textbox(
                label="Goal (edit freely or type your own)",
                placeholder="Type your high-level goal here...",
                lines=4,
                interactive=True,
            )

            moderation_cb = gr.Checkbox(
                label="Enable Moderation Guardrails",
                value=False,
                info="Pre-flight checks goal text; post-flight checks AI output."
            )

        with gr.Column(scale=1):
            gr.Markdown("### Actions")
            run_btn   = gr.Button("▶ Run Engine",   variant="primary",   elem_id="run-btn")
            clear_btn = gr.Button("✕ Clear",         variant="secondary", elem_id="clear-btn")

            gr.Markdown("""
            **Domains:** Legal · Marketing

            **Agents:** Librarian · Researcher · Writer

            **Guardrails:** Pre-flight · Post-flight

            **Analytics:** Token in/out, execution time
            """)

    gr.Markdown("---")

    # ── Status Bar ────────────────────────────────────────────────────────────
    status_md = gr.Markdown("")

    # ── Results Zone ──────────────────────────────────────────────────────────
    gr.Markdown("### Trace Dashboard")
    result_html = gr.HTML(label="Engine Output")

    # ── Glossary / Reference ──────────────────────────────────────────────────
    with gr.Accordion("📚 Reference: Architecture Glossary", open=False):
        gr.Markdown("""
        | Term | Definition |
        |---|---|
        | **Glass Box** | An engine with fully auditable, transparent reasoning steps — no hidden logic |
        | **Control Deck** | The configuration layer that sets the engine's goal, model, and data namespaces |
        | **Context Chaining** | Passing the output of one agent step as input to the next via `
STEP_N_OUTPUT
` tokens |
        | **High-Fidelity RAG** | Retrieval-Augmented Generation with source citations, enabling verifiable answers |
        | **ContextLibrary** | Pinecone namespace holding operational instructions and domain-specific context rules |
        | **KnowledgeStore** | Pinecone namespace holding the actual documents (legal, marketing, etc.) |
        | **Librarian Agent** | Retrieves context instructions from `ContextLibrary` to configure downstream agents |
        | **Researcher Agent** | Performs RAG queries against `KnowledgeStore`, enforces sanitization on retrieved chunks |
        | **Writer Agent** | Synthesizes a final coherent response from all previous agents' outputs |
        | **Pre-flight Moderation** | Safety check on user input *before* the engine runs |
        | **Post-flight Moderation** | Safety check on AI output *before* it is displayed to the user |
        | **Token Analytics** | Per-step tracking of input/output tokens for cost-efficiency monitoring |
        """)

    with gr.Accordion("💡 How to Use This UI", open=False):
        gr.Markdown("""
        ### Quick Start

        1. **Select a preset** from the dropdown — it auto-fills the goal box
        2. **Edit the goal** freely — the text box is fully editable
        3. Optionally **enable Moderation Guardrails**
        4. Click **▶ Run Engine** and wait for the trace dashboard

        ### Understanding the Results

        - Each **step card** shows one agent invocation. Click the header to collapse/expand.
        - The **▶ View Resolved Source Data** link expands the raw JSON context that was passed to the agent.
        - **Token pills** (📥 IN / 📤 OUT) show per-step model usage.
        - The **Final Orchestration Result** (green card at the bottom) is the engine's consolidated answer.

        ### Proving Domain-Agnosticism

        Run preset **#1 or #2** (Marketing), then run **#4 or #5** (Legal). Notice that:
        - The **same three agents** are used in both domains
        - The **same planner logic** generates the execution plan
        - The **same Pinecone index** holds both domains' documents
        - Only the **goal text** drives the different behavior

        This is the core thesis of *Context Engineering for Multi-Agent Systems*.
        """)

    # ── Event Bindings ────────────────────────────────────────────────────────
    preset_dd.change(
        fn=load_preset,
        inputs=[preset_dd],
        outputs=[goal_box]
    )

    run_btn.click(
        fn=run_engine,
        inputs=[goal_box, moderation_cb],
        outputs=[status_md, result_html]
    )

    clear_btn.click(
        fn=lambda: ("── Select a preset goal ──", "", False, "", ""),
        outputs=[preset_dd, goal_box, moderation_cb, status_md, result_html]
    )


# ── Launch ────────────────────────────────────────────────────────────────────
print("Launching Gradio app...")
demo.launch(
    share=True,         # Creates a public tunnel URL — share with colleagues!
    debug=False,        # Set True to see Python tracebacks in the browser
    quiet=True,
)

/tmp/ipykernel_884/2023344046.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Launching Gradio app...
* Running on public URL: https://51d3009d38539ee77d.gradio.live
